# 03 · Agent 搭建：大模型连接与 ReAct 基线

1. 读取 `keys.py`，连接兼容 OpenAI 接口的大模型（默认 DeepSeek）。
2. 搭一个 ReAct agent：每一轮把全部工具定义和历史发给大模型，由它决定下一步调用哪个工具。
3. 在两条任务上运行，看调用轨迹、token 和耗时，并用大模型裁判对照参考答案打分。

**会调用大模型，产生少量费用。** 所有请求都缓存在 `cache/llm/`，重跑相同的请求不再计费。

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import pandas as pd
from jif import config, finmcp
from jif.llm import LLM, Ledger
from jif.mcp_conn import McpConnection
from jif.agents import ReActAgent, judge
pd.set_option("display.max_colwidth", 120)

keys = config.load_keys()
print(config.describe(keys))
ledger = Ledger()
llm = LLM(keys["LLM_BASE_URL"], keys["LLM_API_KEY"], keys["LLM_MODEL"], ledger=ledger)

In [ ]:
try:
    print("可用模型：", llm.list_models()[:20])
except Exception as e:
    print("无法列出模型：", e)
r = llm.chat([{"role": "user", "content": "用一句话说明什么是基金定投。"}], tag="ping")
print(r["message"]["content"], r["usage"], f"{r['latency']:.2f}s")

In [ ]:
conn = McpConnection()
react = ReActAgent(llm, conn, max_rounds=12, obs_cap=30000)
print("ReAct 每轮发送的工具数：", len(react.tools))

In [ ]:
rows, runs = [], {}
for tid in ["multi_tool:30", "multi_tool:1"]:
    task = finmcp.get_task(tid)
    q = finmcp.first_question(task)
    r = react.run(q, tid, {"userId": finmcp.user_id(task)})
    j = judge(llm, q, finmcp.reference_answer(task), r["answer"])
    runs[tid] = r
    rows.append({"任务": tid, "问题": q[:40], "轮数": r["rounds"], "大模型调用": r["llm_calls"], "输入 token": r["llm_in"],
                 "输出 token": r["llm_out"], "工具调用": r["tool_calls"], "回放": dict(pd.Series([e["status"] for e in r["replay"]]).value_counts()),
                 "耗时(s)": r["wall"], "裁判": j["score"], "理由": j["reason"]})
pd.DataFrame(rows)

In [ ]:
r = runs["multi_tool:30"]
for t in r["trace"]:
    print(f"{t['name']}({json.dumps(t['args'], ensure_ascii=False)[:120]}) ok={t['ok']} {t['chars']} 字符")
print("\n回答：", r["answer"][:800])

In [ ]:
display(pd.DataFrame(Ledger.summarize(ledger.records)).T)
conn.close()